# Intermediate 07 — Risk, Assurance & Step-Up Authorization for Agents

## Scenario

A Claims Agent acts on behalf of Alice.

The same agent may:

```text
search documentation
read a claim
update a claim
create a $300 payment
attempt a $50,000 payment
```

We build a policy engine that asks:

```text
How risky is this exact action?
What human assurance exists?
How fresh is it?
Is the workload attested?
Is the task valid?
Is approval required?
Should we allow, constrain, step-up or deny?
```


In [ ]:
from dataclasses import dataclass, asdict
from datetime import datetime, timedelta, timezone
from enum import Enum
import copy, json, uuid

def utcnow():
    return datetime.now(timezone.utc)

class Response(str,Enum):
    ALLOW="allow"
    CONSTRAIN="constrain"
    STEP_UP="step_up"
    DENY="deny"


## 1 — Separate human and workload assurance

In [ ]:
human={
 "user":"alice",
 "aal":2,                 # course representation of NIST AAL2
 "authenticated_at":utcnow()-timedelta(minutes=45),
 "device_compliant":True,
 "account_risk":"low"
}

workload={
 "agent":"claims-agent",
 "spiffe_id":"spiffe://corp.example/prod/agents/claims",
 "attested":True,
 "image_approved":True,
 "status":"active"
}

print(json.dumps({"human":human,"workload":workload},indent=2,default=str))


NIST AAL is used for the human authentication dimension. `workload.attested` is a separate machine assurance dimension; it is **not** an “agent AAL.”

## 2 — Action catalog

In [ ]:
ACTIONS={
 "faq.search":{"base":5,"tier":"R1","side_effect":False},
 "claim.read":{"base":15,"tier":"R2","side_effect":False},
 "claim.update":{"base":35,"tier":"R3","side_effect":True},
 "payment.create":{"base":55,"tier":"R4","side_effect":True},
 "audit.disable":{"base":100,"tier":"PROHIBITED","side_effect":True},
}


## 3 — Risk signals

In [ ]:
@dataclass
class RiskSignals:
    action: str
    data_classification: str="internal"
    amount: float=0
    new_beneficiary: bool=False
    auth_age_minutes: int=0
    device_compliant: bool=True
    workload_attested: bool=True
    image_approved: bool=True
    delegation_depth: int=0
    anomaly_score: int=0
    prompt_injection_score: float=0.0


## 4 — Interpretable risk engine

In [ ]:
def score_risk(s:RiskSignals):
    score=ACTIONS[s.action]["base"]
    reasons=[f"base={score}"]

    if s.data_classification=="confidential":
        score+=10; reasons.append("confidential +10")
    elif s.data_classification=="restricted":
        score+=20; reasons.append("restricted +20")

    if s.amount>=10000:
        score+=30; reasons.append("amount>=10000 +30")
    elif s.amount>=1000:
        score+=20; reasons.append("amount>=1000 +20")
    elif s.amount>=500:
        score+=10; reasons.append("amount>=500 +10")

    if s.new_beneficiary:
        score+=15; reasons.append("new beneficiary +15")
    if s.auth_age_minutes>30:
        score+=10; reasons.append("stale auth +10")
    if not s.device_compliant:
        score+=20; reasons.append("device noncompliant +20")
    if not s.workload_attested:
        score+=30; reasons.append("workload unattested +30")
    if not s.image_approved:
        score+=20; reasons.append("image unapproved +20")

    score+=min(20,s.delegation_depth*5)
    if s.delegation_depth:
        reasons.append(f"delegation +{min(20,s.delegation_depth*5)}")

    score+=min(20,s.anomaly_score)
    if s.anomaly_score:
        reasons.append(f"anomaly +{min(20,s.anomaly_score)}")

    if s.prompt_injection_score>=0.8:
        score+=25; reasons.append("prompt injection +25")

    return min(score,100),reasons


## 5 — Compare transactions

In [ ]:
cases=[
 RiskSignals("claim.read",auth_age_minutes=45),
 RiskSignals("payment.create",amount=100,auth_age_minutes=5),
 RiskSignals("payment.create",amount=5000,new_beneficiary=True,auth_age_minutes=45),
]
for c in cases:
    print(c.action,c.amount,score_risk(c))


## 6 — Required assurance policy

In [ ]:
def requirements(action,risk):
    if action=="audit.disable":
        return {"prohibited":True}

    if risk < 30:
        return {"min_aal":1,"max_auth_age":480,"approval":False,"attested":True}
    if risk < 60:
        return {"min_aal":2,"max_auth_age":60,"approval":False,"attested":True}
    if risk < 80:
        return {"min_aal":2,"max_auth_age":5,"approval":True,"attested":True}
    return {"deny":True}

for risk in [10,40,70,90]:
    print(risk,requirements("payment.create",risk))


## 7 — Decision engine

In [ ]:
def decide(signals,human,workload,approval=None):
    risk,reasons=score_risk(signals)
    req=requirements(signals.action,risk)

    if req.get("prohibited"):
        return Response.DENY,risk,reasons+["absolute deny ceiling"]
    if req.get("deny"):
        return Response.DENY,risk,reasons+["risk exceeds ceiling"]
    if req["attested"] and not workload["attested"]:
        return Response.STEP_UP,risk,reasons+["fresh workload assurance required"]
    if human["aal"] < req["min_aal"]:
        return Response.STEP_UP,risk,reasons+[f'AAL{req["min_aal"]} required']

    auth_age=(utcnow()-human["authenticated_at"]).total_seconds()/60
    if auth_age > req["max_auth_age"]:
        return Response.STEP_UP,risk,reasons+[f'fresh auth <= {req["max_auth_age"]}m required']

    if req["approval"] and not (approval and approval.get("valid")):
        return Response.STEP_UP,risk,reasons+["approval required"]

    if risk>=30:
        return Response.CONSTRAIN,risk,reasons+["bounded authority"]
    return Response.ALLOW,risk,reasons

print(decide(RiskSignals("claim.read",auth_age_minutes=45),human,workload))


## 8 — Authentication freshness

In [ ]:
s=RiskSignals("payment.create",amount=300,auth_age_minutes=45)
print(decide(s,human,workload))


## 9 — RFC 9470-style challenge

In [ ]:
def stepup_challenge(required_acr="urn:example:aal2",max_age=300):
    return {
      "status":401,
      "error":"insufficient_user_authentication",
      "authentication_required":{
        "acr_values":[required_acr],
        "max_age":max_age
      }
    }

print(json.dumps(stepup_challenge(),indent=2))


The notebook uses a simplified Python object to teach the protocol concept. In production, implement RFC 9470 exactly through your OAuth resource/client/authorization-server stack.

## 10 — Simulate fresh authentication

In [ ]:
fresh_human=copy.deepcopy(human)
fresh_human["authenticated_at"]=utcnow()
print(decide(s,fresh_human,workload))


## 11 — Human approval

In [ ]:
approval={
 "id":"apr:92",
 "valid":True,
 "action":"payment.create",
 "claim":"claim:483",
 "amount":300,
 "beneficiary":"vendor:17",
 "expires_at":utcnow()+timedelta(minutes=5)
}

print(decide(s,fresh_human,workload,approval))


## 12 — Parameter-bound approval

In [ ]:
def approval_matches(a,action,claim,amount,beneficiary):
    return (
      a["valid"]
      and utcnow()<a["expires_at"]
      and a["action"]==action
      and a["claim"]==claim
      and a["amount"]==amount
      and a["beneficiary"]==beneficiary
    )

print(approval_matches(approval,"payment.create","claim:483",300,"vendor:17"))
print(approval_matches(approval,"payment.create","claim:483",900,"vendor:17"))


## 13 — New-beneficiary risk

In [ ]:
known=RiskSignals("payment.create",amount=300,new_beneficiary=False,auth_age_minutes=1)
new=RiskSignals("payment.create",amount=300,new_beneficiary=True,auth_age_minutes=1)
print("known:",score_risk(known))
print("new:",score_risk(new))


## 14 — Workload assurance failure

In [ ]:
bad_workload=copy.deepcopy(workload)
bad_workload["attested"]=False

print(decide(
 RiskSignals("claim.update",workload_attested=False,auth_age_minutes=1),
 fresh_human,
 bad_workload
))


## 15 — Workload re-attestation

In [ ]:
def reattest(workload,observed):
    w=copy.deepcopy(workload)
    required={
      "namespace":"prod",
      "service_account":"claims-agent",
      "image_approved":True
    }
    w["attested"]=all(observed.get(k)==v for k,v in required.items())
    w["image_approved"]=observed.get("image_approved",False)
    return w

observed={
 "namespace":"prod",
 "service_account":"claims-agent",
 "image_approved":True
}
restored=reattest(bad_workload,observed)
print(restored)


## 16 — Progressive autonomy

In [ ]:
def autonomy_level(risk):
    if risk<20:
        return "autonomous"
    if risk<40:
        return "autonomous_with_limits"
    if risk<60:
        return "recommend_or_bounded_action"
    if risk<80:
        return "human_approval_required"
    return "prohibited_or_pause"

for r in [10,30,50,70,90]:
    print(r,autonomy_level(r))


## 17 — Autonomy budget

In [ ]:
budget={
 "max_payment":500,
 "max_external_messages":3,
 "allowed_classifications":{"public","internal"},
 "max_delegation_depth":1
}

def within_budget(signals,budget):
    if signals.amount>budget["max_payment"]:
        return False,"payment budget exceeded"
    if signals.data_classification not in budget["allowed_classifications"]:
        return False,"classification outside budget"
    if signals.delegation_depth>budget["max_delegation_depth"]:
        return False,"delegation depth exceeded"
    return True,"within budget"

print(within_budget(RiskSignals("payment.create",amount=300),budget))
print(within_budget(RiskSignals("payment.create",amount=900),budget))


## 18 — Delegation risk

In [ ]:
for depth in range(5):
    s=RiskSignals("claim.update",delegation_depth=depth,auth_age_minutes=1)
    print("depth",depth,"risk",score_risk(s)[0])


## 19 — Prompt injection as enforcement signal

In [ ]:
normal=RiskSignals("claim.update",prompt_injection_score=.1,auth_age_minutes=1)
injected=RiskSignals("claim.update",prompt_injection_score=.95,auth_age_minutes=1)

print("normal:",score_risk(normal))
print("injected:",score_risk(injected))
print("response:",decide(injected,fresh_human,workload))


## 20 — Dynamic tool set

In [ ]:
TOOLS={
 "faq.search":"R1",
 "claim.read":"R2",
 "claim.update":"R3",
 "payment.create":"R4"
}

def tools_for_risk(current_risk):
    if current_risk>=80:
        return []
    if current_risk>=60:
        return ["faq.search","claim.read"]
    if current_risk>=40:
        return ["faq.search","claim.read","claim.update"]
    return list(TOOLS)

for r in [10,50,70,90]:
    print(r,tools_for_risk(r))


## 21 — Absolute deny ceiling

In [ ]:
print(decide(
 RiskSignals("audit.disable"),
 {"user":"alice","aal":3,"authenticated_at":utcnow(),"device_compliant":True},
 {"agent":"claims-agent","attested":True,"image_approved":True,"status":"active"},
 {"valid":True}
))


## 22 — Step-up token design

In [ ]:
stepup_authority={
 "aud":"https://payments.example",
 "scope":["payment:create"],
 "task":"task:483",
 "claim":"claim:483",
 "max_amount":300,
 "expires_in_seconds":120,
 "human_auth_fresh":True,
 "workload_attested":True
}
print(json.dumps(stepup_authority,indent=2))


## 23 — Decision evidence

In [ ]:
def evidence(signals,human,workload,approval=None):
    response,risk,reasons=decide(signals,human,workload,approval)
    return {
      "decision_id":str(uuid.uuid4()),
      "action":signals.action,
      "risk_score":risk,
      "response":response.value,
      "human_aal":human["aal"],
      "auth_age_seconds":int((utcnow()-human["authenticated_at"]).total_seconds()),
      "workload_attested":workload["attested"],
      "approval_id":approval.get("id") if approval else None,
      "reasons":reasons
    }

print(json.dumps(evidence(
 RiskSignals("payment.create",amount=300,auth_age_minutes=1),
 fresh_human,workload,approval
),indent=2))


## 24 — Adversarial bypass tests

In [ ]:
# 1. Strong authentication cannot bypass an absolute deny.
r=decide(
 RiskSignals("audit.disable"),
 {"user":"alice","aal":3,"authenticated_at":utcnow(),"device_compliant":True},
 workload,
 {"valid":True}
)[0]
assert r==Response.DENY

# 2. High-value transaction cannot be made safe by merely being logged in.
r=decide(
 RiskSignals("payment.create",amount=50000,auth_age_minutes=1),
 fresh_human,workload,{"valid":True}
)[0]
assert r==Response.DENY

# 3. Unattested workload triggers stronger response.
r=decide(
 RiskSignals("claim.update",workload_attested=False,auth_age_minutes=1),
 fresh_human,bad_workload
)[0]
assert r==Response.STEP_UP

# 4. Approval parameter substitution fails.
assert not approval_matches(
 approval,"payment.create","claim:483",900,"vendor:17"
)

print("all bypass tests passed")


## 25 — Exercise: connect a real OAuth provider

Implement RFC 9470 with an OAuth/OIDC provider that supports appropriate authentication-context/freshness controls.

Test:

```text
resource detects insufficient authentication
-> challenge
-> client requests step-up
-> user authenticates
-> new token
-> resource re-evaluates
```

Document exactly what your provider's `acr` values mean. Do not assume they map to NIST AAL without evidence.


## 26 — Exercise: SPIFFE/SPIRE

Deploy SPIRE locally or in Kubernetes.

Register:

```text
spiffe://example.org/prod/claims-agent
```

Use workload selectors such as:

```text
namespace
service account
container/image properties where available
```

Retrieve an SVID through the Workload API and replace the notebook's boolean `attested` flag with verified workload identity evidence.


## 27 — Exercise: OPA

Use:

```text
policies/opa/risk_stepup.rego
```

Move the response policy into OPA.

Keep risk inputs explicit:

```text
risk score
action
human AAL
auth age
workload attestation
approval
```

Return structured reason codes.


## 28 — Exercise: Cedar

Use:

```text
policies/cedar/risk_stepup.cedar
```

Model the same controls with typed principals/actions/resources/context.

Compare:

```text
Rego -> flexible policy over arbitrary structured input
Cedar -> typed authorization policy model
```


## 29 — Exercise: progressive autonomy

Create an agent with five levels:

```text
L0 observe
L1 recommend
L2 low-risk autonomous
L3 bounded write
L4 approved high-impact action
```

Design policy that can move the agent **down** as well as up based on current risk.


## 30 — Exercise: approval UX

Design an approval object and UI payload for:

```text
payment.create
```

The approver must see:

```text
agent
user
task
claim
amount
beneficiary
reason
risk signals
data sources
expiration
```

Then attempt parameter substitution after approval.


## 31 — Review questions

1. What is the difference between risk and assurance?
2. What are IAL, AAL and FAL?
3. How many NIST authentication assurance levels exist?
4. Why should workload assurance not be called agent AAL?
5. What evidence can SPIFFE/SPIRE provide?
6. What is workload attestation?
7. Why does AAL3 not automatically authorize a payment?
8. What is RFC 9470?
9. What does `insufficient_user_authentication` communicate?
10. What are `acr`, `auth_time`, `acr_values` and `max_age`?
11. What is the difference between step-up authentication and step-up authorization?
12. What can workload step-up mean?
13. Why are transaction parameters risk inputs?
14. What is progressive autonomy?
15. What is an autonomy budget?
16. Why should approval be parameter-bound?
17. How can prompt-injection detection affect authorization?
18. Why does delegation depth affect risk?
19. What is an absolute deny ceiling?
20. Why should post-step-up credentials be narrow and short-lived?
21. Why must assurance evidence have freshness requirements?
22. What should be captured in a risk authorization decision record?

# Next course

## Intermediate 08 — Workload Assurance & Runtime Attestation for Agents
